# CAP4630 - Intro to Artificial Intelligence

## An Experimental Study of CNN-Based Traffic Sign Recognition for Autonomous Driving Scenarios

By: Jonathan Yunes

This project looks at how well a convolutional neural network (CNN) can recognize and classify traffic signs from images.  The main goal is to test how the model performs under different conditions like changes in lighting and image quality.

In [ ]:
# Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# These imports are for deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# This import is for image processing
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Import for evaluation
from sklearn.metrics import classification_report, confusion_matrix

## Extracts Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Unzips the dataset
import zipfile

zip_path = "/content/drive/MyDrive/traffic sign data/archive.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/data')


Mounted at /content/drive


## Loads the dataset

In [ ]:
data_path = "/content/data/db_lisa_tiny"
files = os.listdir(data_path)

print("Number of files:", len(files))
print(files[:10])

Number of files: 901
['sample_314.png', 'sample_815.png', 'sample_548.png', 'sample_296.png', 'sample_346.png', 'sample_841.png', 'sample_393.png', 'sample_081.png', 'sample_411.png', 'sample_545.png']


## Loads the labels

In [ ]:
csv_path = os.path.join(data_path, "annotations.csv")
df = pd.read_csv(csv_path)

df.head()

,filename,x1,y1,x2,y2,class
0,sample_001.png,190,40,211,63,stop
1,sample_002.png,4,246,43,283,stop
2,sample_003.png,389,286,418,314,stop
3,sample_004.png,307,243,315,251,stop
4,sample_005.png,377,249,398,270,stop


## Explores the Dataset


In [ ]:
# Checks the first row
df.head()

,filename,x1,y1,x2,y2,class
0,sample_001.png,190,40,211,63,stop
1,sample_002.png,4,246,43,283,stop
2,sample_003.png,389,286,418,314,stop
3,sample_004.png,307,243,315,251,stop
4,sample_005.png,377,249,398,270,stop


In [ ]:
# This checks class names and counts
print("Number of images:", len(df))
print("Number of classes:", df['class'].nunique())

df['class'].value_counts()

Number of images: 900
Number of classes: 9


,count
class,
stop,210
speedLimit35,110
keepRight,110
signalAhead,100
merge,100
pedestrianCrossing,100
speedLimit25,80
yield,45
yieldAhead,45


## Prepares Dataset for CNN

Organizes the images into class folders so our CNN model canload them easily.

In [ ]:
import shutil

source_dir = "/content/data/db_lisa_tiny"
output_dir = "/content/lisa_processed"

os.makedirs(output_dir, exist_ok=True)

for _, row in df.iterrows():
    filename = row["filename"]
    label = str(row["class"])

    class_folder = os.path.join(output_dir, label)
    os.makedirs(class_folder, exist_ok=True)

    src_path = os.path.join(source_dir, filename)
    dst_path = os.path.join(class_folder, filename)

    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)

print("Dataset organized into class folders.")
print(os.listdir(output_dir))

Dataset organized into class folders.
['keepRight', 'signalAhead', 'pedestrianCrossing', 'stop', 'yield', 'speedLimit25', 'merge', 'yieldAhead', 'speedLimit35']


## Loads the Prepared Dataset

The processed dataset is loaded using an image data generator.  The images are resized and normailized, and the dataset is split into the training and validation sets.

In [ ]:
# These are the image settings
img_size = (64, 64)
batch_size = 32

# Creates the image generator
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# This is the training data
train_data = datagen.flow_from_directory(
    "/content/lisa_processed",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

# This is the validation data
val_data = datagen.flow_from_directory(
    "/content/lisa_processed",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)

Found 720 images belonging to 9 classes.
Found 180 images belonging to 9 classes.


## Experiment 1: Baseline CNN Model

For this experiment, I made a basic CNN model and it's trained on the prepared traffic sign dataset.  This model is being used as the starting points so the results can be compared.

In [ ]:
# This is the number of classes
num_classes = train_data.num_classes

# Builds the baseline CNN model
baseline_model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

# Compiles the model
baseline_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Shows the model structure
baseline_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,626,313 (6.20 MB)

 Trainable params: 1,626,313 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Trains the CNN model
baseline_history = baseline_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 16s 517ms/step - accuracy: 0.3097 - loss: 1.9240 - val_accuracy: 0.4389 - val_loss: 1.4823
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 7s 314ms/step - accuracy: 0.5736 - loss: 1.1872 - val_accuracy: 0.5889 - val_loss: 1.0856
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 380ms/step - accuracy: 0.7583 - loss: 0.7236 - val_accuracy: 0.6944 - val_loss: 0.7058
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 363ms/step - accuracy: 0.8542 - loss: 0.4088 - val_accuracy: 0.8000 - val_loss: 0.5415
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 334ms/step - accuracy: 0.9083 - loss: 0.2533 - val_accuracy: 0.7944 - val_loss: 0.5215
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 376ms/step - accuracy: 0.9500 - loss: 0.1670 - val_accuracy: 0.8278 - val_loss: 0.4505
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 7s 310ms/step - accuracy: 0.9472 - loss: 0.1544 - val_accuracy: 0.8667 - val_loss: 0.3661
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 379ms/step - accuracy: 0.9625 - loss: 0.0954 - val_accuracy: 0

In [ ]:
# Evaluation of the baseline model
val_loss, val_acc = baseline_model.evaluate(val_data)

print("Validation Accuracy:", val_acc)
print("Validation Loss:", val_loss)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 345ms/step - accuracy: 0.8389 - loss: 0.4301
Validation Accuracy: 0.8388888835906982
Validation Loss: 0.4301091134548187


## Experiment 2: CNN with Data Augmentation

For this experiment,  data augmentation is being applied to the dataset to improve the model's performance.

In [ ]:
# Creates the data augmentation generator
aug_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

train_aug = aug_datagen.flow_from_directory(
    "/content/lisa_processed",
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_aug = aug_datagen.flow_from_directory(
    "/content/lisa_processed",
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 720 images belonging to 9 classes.
Found 180 images belonging to 9 classes.


In [ ]:
# This rebuilds the model
aug_model = keras.models.clone_model(baseline_model)

aug_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Trains the model with augmented data
aug_history = aug_model.fit(
    train_aug,
    validation_data=val_aug,
    epochs=10
)

Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 14s 469ms/step - accuracy: 0.1875 - loss: 2.1295 - val_accuracy: 0.2444 - val_loss: 1.8994
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 425ms/step - accuracy: 0.3333 - loss: 1.8523 - val_accuracy: 0.3444 - val_loss: 1.7619
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 417ms/step - accuracy: 0.3708 - loss: 1.6896 - val_accuracy: 0.3889 - val_loss: 1.6828
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 354ms/step - accuracy: 0.3958 - loss: 1.6003 - val_accuracy: 0.4000 - val_loss: 1.5526
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 418ms/step - accuracy: 0.4222 - loss: 1.5448 - val_accuracy: 0.3944 - val_loss: 1.5318
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 385ms/step - accuracy: 0.4556 - loss: 1.4898 - val_accuracy: 0.4556 - val_loss: 1.4774
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 379ms/step - accuracy: 0.5097 - loss: 1.3962 - val_accuracy: 0.4222 - val_loss: 1.4832
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 418ms/step - accuracy: 0.4903 - loss: 1.3896 - val_accuracy

In [ ]:
# Evaluates
val_loss_aug, val_acc_aug = aug_model.evaluate(val_aug)

print("Augmented Validation Accuracy:", val_acc_aug)
print("Augmented Validation Loss:", val_loss_aug)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 265ms/step - accuracy: 0.5333 - loss: 1.2879
Augmented Validation Accuracy: 0.5333333611488342
Augmented Validation Loss: 1.287882685661316


## Experiment 3: CNN Under Challenging Conditions

The CNN model is tested using images with difficult conditions.  The images are modified to simulate challenges like lower brightness and image quality.

In [ ]:
# This creates a generator for challenging image conditions
challenging_datagen = ImageDataGenerator(
    rescale=1./255,
    brightness_range=[0.4, 0.8],
    zoom_range=0.2,
    validation_split=0.2
)

train_challenge = challenging_datagen.flow_from_directory(
    "/content/lisa_processed",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_challenge = challenging_datagen.flow_from_directory(
    "/content/lisa_processed",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)

Found 720 images belonging to 9 classes.
Found 180 images belonging to 9 classes.


In [ ]:
# Rebuilds the model again
challenge_model = keras.models.clone_model(baseline_model)

challenge_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Trains the model under challenging conditions
challenge_history = challenge_model.fit(
    train_challenge,
    validation_data=val_challenge,
    epochs=10
)

Epoch 1/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 15s 537ms/step - accuracy: 0.2708 - loss: 2.0053 - val_accuracy: 0.3444 - val_loss: 1.7523
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 15s 353ms/step - accuracy: 0.4347 - loss: 1.5477 - val_accuracy: 0.5056 - val_loss: 1.3206
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 428ms/step - accuracy: 0.6306 - loss: 1.0843 - val_accuracy: 0.6000 - val_loss: 1.0237
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 429ms/step - accuracy: 0.7278 - loss: 0.8249 - val_accuracy: 0.6556 - val_loss: 0.8959
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 357ms/step - accuracy: 0.7625 - loss: 0.6721 - val_accuracy: 0.7278 - val_loss: 0.6860
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 423ms/step - accuracy: 0.8194 - loss: 0.5749 - val_accuracy: 0.7333 - val_loss: 0.7116
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 525ms/step - accuracy: 0.8486 - loss: 0.4480 - val_accuracy: 0.8000 - val_loss: 0.6312
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 356ms/step - accuracy: 0.8778 - loss: 0.3936 - val_accura

In [ ]:
# Evaluates the model
val_loss_challenge, val_acc_challenge = challenge_model.evaluate(val_challenge)

print("Challenging Conditions Validation Accuracy:", val_acc_challenge)
print("Challenging Conditions Validation Loss:", val_loss_challenge)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 266ms/step - accuracy: 0.8278 - loss: 0.4790
Challenging Conditions Validation Accuracy: 0.8277778029441833
Challenging Conditions Validation Loss: 0.4789925515651703
